In [3]:
import pandas as pd
import numpy as np
import re
from IPython.display import display

print("=== TAHAP 1: DATA PREPROCESSING (MURNI KLASIFIKASI SENTIMEN) ===")

# 1. LOAD DATA 
print("1. Membaca data FOMC & Bitcoin...")
df_fomc = pd.read_csv(r"D:\Collage\AnuJurnal\dataset\teksgacor_powelthefed\01_raw_thefed.csv")
df_btc = pd.read_csv(r"D:\Collage\AnuJurnal\dataset\datasetbtc\01_raw_btc.csv", sep=';', decimal=',') 

# 2. STANDARISASI TANGGAL
print("2. Menyamakan format tanggal...")
if 'Tanggal' in df_fomc.columns: df_fomc['Tanggal'] = pd.to_datetime(df_fomc['Tanggal'], errors='coerce')
if 'Date' in df_btc.columns: df_btc.rename(columns={'Date': 'Tanggal'}, inplace=True)
df_btc['Tanggal'] = pd.to_datetime(df_btc['Tanggal'], errors='coerce')

# 3. LABELING TARGET SENTIMEN (Berdasarkan Reaksi Pasar)
print("3. Membuat Label Target Sentimen Teks...")
df_btc = df_btc.sort_values('Tanggal').reset_index(drop=True)
df_btc['Close_Besok'] = df_btc['Close'].shift(-1)

# === MODIFIKASI KRUSIAL: LABEL LANGSUNG BERUPA TEKS (KATEGORI) ===
# Jika pasar naik -> Sentimen The Fed direspons Positif (Dovish)
# Jika pasar turun -> Sentimen The Fed direspons Negatif (Hawkish)
df_btc['Sentimen_Aktual'] = np.where(df_btc['Close_Besok'] > df_btc['Close'], 'Positif (Dovish)', 'Negatif (Hawkish)')
df_btc = df_btc.dropna(subset=['Close_Besok'])

# 4. PENGGABUNGAN & PENGHAPUSAN KOLOM HARGA NOMINAL
print("4. Menggabungkan data & membuang kolom harga nominal...")
df_gabungan = pd.merge(df_fomc, df_btc[['Tanggal', 'Sentimen_Aktual']], on='Tanggal', how='inner')

if len(df_gabungan) == 0:
    print(" WARNING: Data gabungan kosong! Cek format tanggal.")
else:
    print(f" Merge Sukses! Tersisa {len(df_gabungan)} baris dokumen.")

# 5. PEMBERSIHAN TEKS
print("5. Membersihkan teks pidato...")
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

if 'Teks_Pidato' in df_gabungan.columns:
    df_gabungan['Teks_Pidato'] = df_gabungan['Teks_Pidato'].apply(clean_text)

# 6. SIMPAN DATASET KLASIFIKASI MURNI
df_gabungan.to_csv('02_dataset_klasifikasi_sentimen.csv', index=False, encoding='utf-8-sig', sep=';')
print(" PREPROCESSING SELESAI! Dataset telah menjadi Dataset Klasifikasi Teks Murni.")

print("\n" + "="*70)
print(" Preview Dataset Klasifikasi Sentimen:")
print("="*70)
display(df_gabungan.head())

=== TAHAP 1: DATA PREPROCESSING (MURNI KLASIFIKASI SENTIMEN) ===
1. Membaca data FOMC & Bitcoin...
2. Menyamakan format tanggal...
3. Membuat Label Target Sentimen Teks...
4. Menggabungkan data & membuang kolom harga nominal...
 Merge Sukses! Tersisa 48 baris dokumen.
5. Membersihkan teks pidato...
 PREPROCESSING SELESAI! Dataset telah menjadi Dataset Klasifikasi Teks Murni.

 Preview Dataset Klasifikasi Sentimen:


,Tanggal,Teks_Pidato,Sentimen_Aktual
0,2020-02-19,"January 28-29, 2020 A joint meeting of the Fed...",Negatif (Hawkish)
1,2020-04-08,"March 15, 2020 A joint meeting of the Federal ...",Negatif (Hawkish)
2,2020-05-20,"April 28–29, 2020A joint meeting of the Federa...",Negatif (Hawkish)
3,2020-07-01,"June 9-10, 2020 A joint meeting of the Federal...",Negatif (Hawkish)
4,2020-08-19,"July 28-29, 2020A joint meeting of the Federal...",Positif (Dovish)


In [1]:
import pandas as pd
import numpy as np
import re

print("=== DATA COLLECTING & PREPROCESSING (TARGET T+3) ===")

# 1. LOAD DATA MENTAH
print("1. Membaca data mentah FOMC & Bitcoin...")
# Sesuaikan path ini dengan lokasi file asli di laptop Anda
path_fomc = r"D:\Collage\AnuJurnal\dataset\teksgacor_powelthefed\01_raw_thefed.csv"
path_btc = r"D:\Collage\AnuJurnal\dataset\datasetbtc\01_raw_btc.csv"

df_fomc = pd.read_csv(path_fomc)
df_btc = pd.read_csv(path_btc, sep=';', decimal=',') 

# 2. STANDARISASI TANGGAL
if 'Tanggal' in df_fomc.columns: df_fomc['Tanggal'] = pd.to_datetime(df_fomc['Tanggal'], errors='coerce')
if 'Date' in df_btc.columns: df_btc.rename(columns={'Date': 'Tanggal'}, inplace=True)
df_btc['Tanggal'] = pd.to_datetime(df_btc['Tanggal'], errors='coerce')

# 3. MENGHITUNG TARGET KLASIFIKASI T+3 (MEDIUM-TERM TREND)
print("2. Mengalkulasi persentase harga Bitcoin di H+3 (T+3)...")
df_btc = df_btc.sort_values('Tanggal').reset_index(drop=True)

# INI KUNCI PERUBAHANNYA: Mengambil harga penutupan 3 hari setelahnya
df_btc['Close_T3'] = df_btc['Close'].shift(-3) 

# Menghitung persentase return dari hari H ke T+3
df_btc['Return_T3_Persen'] = ((df_btc['Close_T3'] - df_btc['Close']) / df_btc['Close']) * 100
df_btc = df_btc.dropna(subset=['Return_T3_Persen'])

# 4. MEMBUAT LABEL GROUND TRUTH
# Jika dalam 3 hari harganya naik = Dovish. Jika turun = Hawkish.
df_btc['Sentimen_Aktual'] = np.where(df_btc['Return_T3_Persen'] > 0, 'Positif (Dovish)', 'Negatif (Hawkish)')

# 5. GABUNGKAN DENGAN TEKS FOMC
print("3. Menggabungkan teks FOMC dengan Label T+3...")
df_gabungan = pd.merge(df_fomc, df_btc[['Tanggal', 'Return_T3_Persen', 'Sentimen_Aktual']], on='Tanggal', how='inner')

# Pembersihan teks dasar
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[\n\r]+', ' ', text)
    return re.sub(r'\s{2,}', ' ', text).strip()

df_gabungan['Teks_Pidato'] = df_gabungan['Teks_Pidato'].apply(clean_text)

# 6. SIMPAN DATASET BARU
nama_file_output = '02_dataset_klasifikasi_t3.csv'
df_gabungan.to_csv(nama_file_output, sep=';', index=False)
print(f"\n BINGO! Dataset T+3 telah berhasil dibuat dan disimpan sebagai: {nama_file_output}")
display(df_gabungan[['Tanggal', 'Return_T3_Persen', 'Sentimen_Aktual']].head())

=== DATA COLLECTING & PREPROCESSING (TARGET T+3) ===
1. Membaca data mentah FOMC & Bitcoin...
2. Mengalkulasi persentase harga Bitcoin di H+3 (T+3)...
3. Menggabungkan teks FOMC dengan Label T+3...

 BINGO! Dataset T+3 telah berhasil dibuat dan disimpan sebagai: 02_dataset_klasifikasi_t3.csv


,Tanggal,Return_T3_Persen,Sentimen_Aktual
0,2020-02-19,0.309288,Positif (Dovish)
1,2020-04-08,-6.476810,Negatif (Hawkish)
2,2020-05-20,-3.294077,Negatif (Hawkish)
3,2020-07-01,-1.038508,Negatif (Hawkish)
4,2020-08-19,-0.650248,Negatif (Hawkish)
